# Fine-tune Moirai-MoE-base on S&P 500 -- date-based split, engineered features, no early stopping

This is a second, differently-configured run of the same fine-tuning pipeline as `finetune_sp500_moirai_moe.ipynb`, with:

1. **Date-based split**: fine-tune on everything **before 2024-10-01**, test on everything **on/after 2024-10-01** (not a fixed day count).
2. **No early stopping**: training always runs the full configured epoch budget, regardless of validation-loss plateaus. The best checkpoint (lowest validation loss) is still the one saved and evaluated.
3. **Engineered features**: `Day` (weekday), `Month`, `Momentum` (10-day), `MovingAverage` (20-day SMA), `Return`, `LogReturn` are all computed during data prep.
4. **Restricted model context**: only `Close`, `Return`, `LogReturn` are ever fed to the model. `Open`/`High`/`Low`/`Volume`/`Day`/`Month`/`Momentum`/`MovingAverage` are computed and saved for reference/plotting only -- uni2ts's simple multivariate data builder has no notion of a covariate-only channel, so "out of context" here means excluded from the model entirely.
5. **Return-space evaluation**: since `Close` (raw price) is non-stationary while `Return`/`LogReturn` are close to stationary, the evaluation reconstructs an implied Close price path from the forecasted `LogReturn` path and compares it against both the actual price and the model's direct `Close`-channel forecast.

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or better).

Uses `bitsandbytes`' 8-bit AdamW (cuts optimizer memory ~4x) plus fp16 mixed precision, same as the first notebook -- these settings were already proven stable on a free T4.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run this cell.")

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Agrim-Nuware/MOIRAI-CODE.git"
if not os.path.isdir("repo"):
    !git clone $REPO_URL repo
%cd repo

In [ ]:
!pip install -q -e '.[notebook]'
!pip install -q bitsandbytes yfinance

# uni2ts pins numpy~=1.26, which downgrades Colab's preinstalled numpy 2.x.
# Colab's preinstalled pandas is built against numpy 2.x, so once numpy is
# downgraded the two are ABI-incompatible ("numpy.dtype size changed").
# Force-reinstall a matching pandas, then restart the runtime so every
# already-imported module (numpy got pulled in transitively by torch above)
# reloads consistently. This cell intentionally crashes/restarts the kernel --
# that's expected, not an error. After it restarts, just continue running
# from the next cell (installed packages and cloned files are unaffected).
!pip install -q --force-reinstall "numpy<2" "pandas>=2.0,<2.3"
import os

os.kill(os.getpid(), 9)

**The cell above deliberately restarts the Colab runtime** (to fix a numpy/pandas
version mismatch). You'll see a "session crashed" / "automatically restarted" notice --
that's expected. Once it restarts, just continue running the cells below in order;
you do **not** need to re-run the clone or pip install cells.

In [ ]:
%cd /content/repo
import numpy as np
import pandas as pd

print("numpy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
with open(".env", "w") as f:
    f.write("CUSTOM_DATA_PATH=dataset/uni2ts_storage\n")
print(open(".env").read())

## 2. Download 20 years of S&P 500 data, engineer features, split at 2024-10-01

In [ ]:
!python dataset/sp500/prepare_oct2024_split_data.py

In [ ]:
import json
import pandas as pd

with open("dataset/sp500/split_info_v2.json") as f:
    split_info = json.load(f)

trainval_df = pd.read_csv("dataset/sp500/sp500_context_trainval.csv", index_col=0, parse_dates=True)
train_length = split_info["train_length"]
date_offset = trainval_df.index[train_length - 1].strftime("%Y-%m-%d")

print("context variates (fed to model):", split_info["context_variates"])
print("train_length:", train_length)
print("lightning val offset:", split_info["lightning_val_offset"])
print("lightning val length:", split_info["lightning_val_length"])
print("final test length (>= cutoff):", split_info["final_test_len"])
print("date_offset for CSV builder:", date_offset)

## 3. Build the uni2ts HF-format dataset (Close, Return, LogReturn only)

In [ ]:
!python -m uni2ts.data.builder.simple SP500V2 dataset/sp500/sp500_context_trainval.csv \
  --dataset_type wide_multivariate --date_offset "{date_offset}" --freq B

## 4. Fine-tune Moirai-MoE-base (full fine-tune, GPU, fp16 + 8-bit AdamW, no early stopping, 25 epochs)

`trainer.callbacks.2.patience=999999` neutralizes the EarlyStopping callback (it's still present
so validation loss is still logged every epoch, it just never fires before `max_epochs` is reached).
The `ModelCheckpoint` callback is untouched, so the checkpoint that gets saved and evaluated is still
the **best** (lowest validation loss) epoch, not necessarily the last one.

`max_epochs=25` (rather than a much larger budget) is a deliberate, evidence-based choice: in the
first notebook's run, validation loss stopped improving by around epoch 13-16 (early stopping fired
at epoch 16 there). Fine-tuning an already-pretrained 935M-param model on one, comparatively small
time series tends to converge fast, so 25 epochs gives a bit of headroom past where v1 plateaued
without burning a large, mostly-wasted epoch budget the way a 60-epoch cap would.

With only 3 context variates (vs. 5 in the first notebook) each packed sample is shorter, but
`train_dataloader.batch_size=16` / `num_workers=0` are kept identical to the first notebook's
proven-stable settings rather than pushed higher, since we can't test free-tier GPU memory headroom
ahead of time.

In [ ]:
lightning_val_offset = split_info["lightning_val_offset"]
lightning_val_length = split_info["lightning_val_length"]

!python -m cli.train \
  -cp conf/finetune \
  exp_name=sp500_v2_full_finetune \
  run_name=run1 \
  tf32=false \
  model=moirai_moe_1.0_R_base \
  model.patch_size=16 \
  model.context_length=512 \
  model.prediction_length=32 \
  model.num_training_steps=1300 \
  model.num_warmup_steps=50 \
  model.finetune_pattern=full \
  model.use_8bit_adam=true \
  model.lr=1e-5 \
  data=sp500 \
  data.dataset=SP500V2 \
  data.patch_size=16 \
  data.context_length=512 \
  data.prediction_length=32 \
  data.mode=M \
  data.train_length={train_length} \
  data.distance=5 \
  val_data=sp500 \
  val_data.dataset=SP500V2_eval \
  val_data.patch_size=16 \
  val_data.context_length=512 \
  val_data.prediction_length=32 \
  val_data.mode=M \
  val_data.offset={lightning_val_offset} \
  val_data.eval_length={lightning_val_length} \
  val_data.distance=32 \
  trainer.max_epochs=25 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.precision=16-mixed \
  trainer.callbacks.2.patience=999999 \
  +trainer.log_every_n_steps=10 \
  train_dataloader.batch_size=16 \
  train_dataloader.num_workers=0 \
  val_dataloader.batch_size=4 \
  val_dataloader.num_workers=0

## 5. Evaluate: zero-shot vs fine-tuned, on the held-out region (>= 2024-10-01)

Also reconstructs an implied Close price path from the fine-tuned model's forecasted LogReturn
path, to check whether forecasting in return-space beats forecasting the raw Close level directly.

In [ ]:
!python dataset/sp500/evaluate_finetuned_v2.py \
  --context_length 512 --prediction_length 32 --num_samples 100

In [ ]:
from IPython.display import Image, display

print("Forecast comparison (Close price -- direct vs reconstructed from LogReturn):")
display(Image("dataset/sp500/results_v2_forecast_plot.png"))
print("\nError comparison by variate:")
display(Image("dataset/sp500/results_v2_metrics_bar.png"))
print("\nFine-tuning loss curve:")
display(Image("dataset/sp500/results_v2_loss_curve.png"))

In [ ]:
import json

with open("dataset/sp500/results_v2_metrics.json") as f:
    results = json.load(f)

zs, ft = results["zero_shot"], results["fine_tuned"]
print(f"{'metric':<22} {'zero-shot':>14} {'fine-tuned':>14}")
print(f"{'Close MAPE':<22} {zs['Close']['mape']:>13.2f}% {ft['Close']['mape']:>13.2f}%")
print(f"{'Close(recon) MAPE':<22} {zs['Close_reconstructed']['mape']:>13.2f}% {ft['Close_reconstructed']['mape']:>13.2f}%")
print(f"{'Return MAE':<22} {zs['Return']['mae']:>14.5f} {ft['Return']['mae']:>14.5f}")
print(f"{'LogReturn MAE':<22} {zs['LogReturn']['mae']:>14.5f} {ft['LogReturn']['mae']:>14.5f}")

## 6. (Optional) Save results back to your GitHub repo

Uncomment and fill in a [personal access token](https://github.com/settings/tokens) if you
want to push the fine-tuned checkpoint and plots back to your repo. Skip this if you'd rather
just download the files from the Colab file browser (left sidebar).

In [ ]:
# GITHUB_TOKEN = ""  # paste a token with repo write access, or leave blank to skip
# if GITHUB_TOKEN:
#     !git add dataset/sp500/results_v2_*.png dataset/sp500/results_v2_metrics.json
#     !git commit -m "Add Colab v2 fine-tuning results"
#     !git push https://$GITHUB_TOKEN@github.com/Agrim-Nuware/MOIRAI-CODE.git HEAD:main